In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import jaccard_score

from utils_correlations import replace_greek, inverse_replace_greek, get_gene_conversion_dict
from utils_name_maps import cytokine_name_map

In [ ]:
FIGDIR = "./figures"

In [ ]:
celltype_combinations = [
    # (mouse, human)
    ('T_cell_CD4', "CD4_T_cell"),
    ('T_cell_CD8', "CD8_T_cell"),
    ('B_cell', "B_cell"),
    ('Treg', 'Treg'),
    ("ILC", "ILC"),
    ("Monocyte", "CD14_Mono"),
    ("Monocyte", "CD16_Mono"),
    ("NK_cell", "CD56-dim_NK"),
    ("NK_cell", "CD56-bright_NK"),
    ("pDC", "pDC"),
    ("cDC1", "cDC"),
    ("cDC2", "cDC"),
]

title_map_short = {
    "T_cell_CD4__CD4 T cell": "CD4 T cell",
    "T_cell_CD8__CD8 T cell": "CD8 T cell",
    "B_cell__B_cell": "B cell",
    "Treg__Treg": "Treg",
    "ILC__ILC": "ILC",
    "Monocyte__CD14_Mono": "CD14 Mono",
    "Monocyte__CD16_Mono": "CD16 Mono",
    "NK_cell__NK CD56low": "NK CD56low",
    "NK_cell__NK CD56hi": "NK CD56hi",
    "pDC__pDC": "pDC",
    "cDC1__cDC": "cDC1",
    "cDC2__cDC": "cDC2",
}

In [ ]:
def get_data_mouse(
    celltype: str | None = None, 
    cytokine: str | None = None, 
    mouse_to_human_dict: dict | None = None,
    subset_to_homologs: bool = True,
) -> pd.DataFrame:
    
    mcd = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/41586_2023_6816_MOESM5_ESM.xlsx", sheet_name=celltype)
    mcd.loc[:, "cytokine"]  = mcd.Cytokine_Str.map(cytokine_name_map)
    mcd = mcd.loc[~mcd.cytokine.isna()]
    if cytokine is not None:
        mcd = mcd.loc[mcd.cytokine == cytokine]
    mcd.rename({"Gene": "gene", "Celltype_Str": "celltype", "Avg_log2FC": "log_fc"}, axis=1, inplace=True)
    mcd.loc[:, "gene_mouse"] = mcd.gene.copy()
    
    if mouse_to_human_dict is not None:
        mcd.loc[:, "gene"] = mcd.loc[:, "gene"].map(mouse_to_human_dict)
        if subset_to_homologs:
            mcd = mcd.loc[~mcd.gene.isna()]
    return mcd

def get_human_data(
    celltype: str | None = None,
    cytokine: str | None = None,
    hcd: pd.DataFrame | None = None,
)-> pd.DataFrame:

    if hcd is None:
        hcd = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_with_blacklisted_genes_removed.csv", index_col=0)
    hcd = hcd.loc[hcd.celltype == celltype]        
    if cytokine is not None:
        hcd = hcd.loc[hcd.cytokine == cytokine]
    return hcd

def jaccard(x, y):
    intersection = np.intersect1d(x, y)
    union = np.union1d(x, y)
    return len(intersection) / len(union)

In [ ]:
human_to_mouse_dict, mouse_to_human_dict = get_gene_conversion_dict()

In [ ]:
print(f"Total #genes that we can map from mouse to human: {len(mouse_to_human_dict)}")

In [ ]:
all_mouse_signature_genes, all_mouse_signature_genes_homologs = [], []
for celltype_combination in tqdm(celltype_combinations):
    (celltype_m, celltype_h) = celltype_combination
    mcd = get_data_mouse(celltype=celltype_m, mouse_to_human_dict=mouse_to_human_dict, subset_to_homologs=False)
    all_mouse_signature_genes.extend(mcd.gene_mouse.unique())
    all_mouse_signature_genes_homologs.extend(mcd.loc[~mcd.gene.isna()].gene_mouse.unique())
all_mouse_signature_genes, all_mouse_signature_genes_occurrences = np.unique(all_mouse_signature_genes, return_counts=True)
all_mouse_signature_genes_homologs, _ = np.unique(all_mouse_signature_genes_homologs, return_counts=True)

In [ ]:
mask = np.isin(all_mouse_signature_genes, all_mouse_signature_genes_homologs)
sns.histplot(all_mouse_signature_genes_occurrences[~mask], bins=range(14), discrete=True)
plt.xticks(range(14))
plt.show()

In [ ]:
print(f"Total number of genes that occur in any signature: {len(all_mouse_signature_genes)}")

In [ ]:
print(f"Total number of homolog genes that occur in any signature: {len(all_mouse_signature_genes_homologs)}")

In [ ]:
# load only once
hcd_all = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_with_blacklisted_genes_removed.csv", index_col=0)

In [ ]:
threshold_p_value = 0.05
threshold_p_logfc = 0.25
threshold_de_wells = 3

scores = pd.DataFrame()

for celltype_combination in tqdm(celltype_combinations):
    (celltype_m, celltype_h) = celltype_combination
    hcd = get_human_data(celltype=celltype_h, hcd=hcd_all)
    mcd = get_data_mouse(celltype=celltype_m, mouse_to_human_dict=mouse_to_human_dict, subset_to_homologs=False)
    shared_cytokines = np.intersect1d(mcd.cytokine.unique(), hcd.cytokine.unique())

    for cytokine in tqdm(shared_cytokines, leave=False):
        hcd_c = hcd.loc[hcd.cytokine == cytokine]
        mcd_signature = mcd.loc[mcd.cytokine == cytokine] # mouse dict only reports significant genes (=signature genes)
        mcd_signature_homologs = mcd_signature.loc[~mcd_signature.gene.isna()]

        hcd_signature = hcd_c.loc[
            (
                (hcd_c.adj_p_value < threshold_p_value) 
                & (hcd_c.log_fc.abs() > threshold_p_logfc)
                & (hcd_c.num_DE_pbs_wells > threshold_de_wells)
                
            )
        ]

        shared_genes_all       = np.intersect1d(mcd_signature_homologs.gene.values, hcd_c.gene.values)
        shared_genes_signature = np.intersect1d(mcd_signature_homologs.gene.values, hcd_signature.gene.values)

        hcd_signature_shared_genes = hcd_signature.set_index("gene").loc[shared_genes_signature]
        mcd_signature_shared_genes = mcd_signature_homologs.set_index("gene").loc[shared_genes_signature]

        hcd_all_shared_genes = hcd_c.set_index("gene").loc[shared_genes_all]
        mcd_all_shared_genes = mcd_signature_homologs.set_index("gene").loc[shared_genes_all]

        
        row_idx = scores.shape[0]
        # meta info
        scores.loc[row_idx, "cytokine"] = cytokine
        scores.loc[row_idx, "celltype_h"] = celltype_h
        scores.loc[row_idx, "celltype_m"] = celltype_m
        scores.loc[row_idx, "celltype_combination"] = "__".join(celltype_combination)
        # gene set size info
        scores.loc[row_idx, "num_signature_genes_m"] = mcd_signature.shape[0]
        scores.loc[row_idx, "num_signature_genes_m_homologs"] = mcd_signature_homologs.shape[0]
        scores.loc[row_idx, "num_signature_genes_h"] = hcd_signature.shape[0]
        scores.loc[row_idx, "num_signature_genes_shared"] = len(shared_genes_signature)
        scores.loc[row_idx, "num_all_genes_shared"] = len(shared_genes_all)
        # stats
        scores.loc[row_idx, "jaccard_signature"] = jaccard(hcd_signature.gene.values, mcd_signature_homologs.gene.values)
        scores.loc[row_idx, "isin_hm_signature"] = np.isin(hcd_signature.gene.values, mcd_signature_homologs.gene.values).sum()
        scores.loc[row_idx, "isin_mh_signature"] = np.isin(mcd_signature_homologs.gene.values, hcd_signature.gene.values).sum()
        if len(shared_genes_signature) > 1:
            scores.loc[row_idx, "pearson_signature_shared"] = pearsonr(hcd_signature_shared_genes.log_fc, mcd_signature_shared_genes.log_fc)[0]
            scores.loc[row_idx, "spearman_signature_shared"] = spearmanr(hcd_signature_shared_genes.log_fc, mcd_signature_shared_genes.log_fc)[0]
        if len(shared_genes_all) > 1:
            scores.loc[row_idx, "pearson_all_shared"] = pearsonr(hcd_all_shared_genes.log_fc, mcd_all_shared_genes.log_fc)[0]
            scores.loc[row_idx, "spearman_all_shared"] = spearmanr(hcd_all_shared_genes.log_fc, mcd_all_shared_genes.log_fc)[0]
    
    # break

In [ ]:
scores

In [ ]:
scores.loc[:, "isin_hm_signature_frac"] = scores.loc[:, "isin_hm_signature"] / scores.loc[:, "num_signature_genes_h"]
scores.loc[:, "isin_mh_signature_frac"] = scores.loc[:, "isin_mh_signature"] / scores.loc[:, "num_signature_genes_m"]
scores.loc[:, "isin_mh_signature_frac_homologs"] = scores.loc[:, "isin_mh_signature"] / scores.loc[:, "num_signature_genes_m_homologs"]

In [ ]:
sns.heatmap(
    scores.pivot(
        columns="celltype_combination",
        index="cytokine",
        values="isin_hm_signature_frac"
    ),
    cmap="mako",
)
plt.show()

In [ ]:
sns.heatmap(
    scores.pivot(
        columns="celltype_combination",
        index="cytokine",
        values="isin_mh_signature_frac"
    ),
    cmap="mako",
)
plt.show()

In [ ]:
_scores = scores.pivot(
    columns="celltype_combination",
    index="cytokine",
    values="isin_mh_signature_frac_homologs"
)

sns.heatmap(
    data=_scores,
    cmap="mako",
)
plt.show()

In [ ]:
mean_per_cytokine

In [ ]:
_scores = scores.pivot(
    columns="celltype_combination",
    index="cytokine",
    values="isin_mh_signature_frac"
)
_scores = _scores.loc[np.sum(_scores.fillna(0)>0.2, axis=1) > 0]
_scores = _scores.loc[:, np.sum(_scores.fillna(0)>0.2, axis=0) > 0]

mean_per_cytokine = _scores.mean(axis=1).sort_values(ascending=False)
# order_y = mean_per_cytokine[mean_per_cytokine>0.1].index
order_y = mean_per_cytokine.index
order_x = _scores.mean(axis=0).sort_values(ascending=False).index

_scores = _scores.loc[order_y, order_x]

fig, ax = plt.subplots(1,2,figsize=(3.5, 3.5), sharey=False, width_ratios=(0.8, 0.2), gridspec_kw={})
ax_hm = ax[0]
ax_bar = ax[1]
sns.heatmap(_scores, square=True, cbar=None, vmin=0, vmax=1, linewidth=0.5, linecolor="white", cmap="Oranges", ax=ax_hm, )
ax_hm.set_facecolor("lightgray")
ax_hm.set_xlabel("")
ax_hm.set_ylabel("")
ax_hm.set_xticks(0.5+np.arange(_scores.shape[1]), [l.get_text().split("__")[1].replace("_", " ") for l in ax_hm.get_xticklabels()], fontsize=6, family="Arial")
ax_hm.set_yticks(0.5+np.arange(_scores.shape[0]), [replace_greek(l) for l in _scores.index], fontsize=6, family="Arial")
ax_hm.tick_params(length=0)

box = ax_bar.get_position()
box.x0 = box.x0 - 0.25
box.x1 = box.x1 - 0.25
ax_bar.set_position(box)

sns.barplot(
    data=scores.loc[scores.cytokine.isin(order_y)],
    y="cytokine",
    x="num_signature_genes_m",
    ax=ax_bar,
    order=order_y,
    color="gold",
    err_kws={'linewidth': 0.5},
)
sns.barplot(
    data=scores.loc[scores.cytokine.isin(order_y)],
    y="cytokine",
    x="num_signature_genes_m_homologs",
    ax=ax_bar,
    order=order_y,
    color="gray",
    err_kws={'linewidth': 0.5},
    alpha=0.6,
    errorbar=None,
)
ax_bar.set_xscale("log")
ax_bar.set_ylabel("")
ax_bar.set_xlabel("")
ax_bar.set_yticks([])
ax_bar.set_xticks([10, 100, 1000], ["10", "100", "1K"], fontsize=6, family="Arial")
ax_bar.set_xlim([0, 1000])
ax_bar.grid("y")
ax_bar.spines[['right', 'top']].set_visible(False)
ax_bar.set_xlabel("# genes", fontsize=6, family="Arial")

plt.savefig(os.path.join(FIGDIR, f"heatmap_dict_overlap_mh_homologs.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
scores

In [ ]:
mouse_color = "#1B9E77"
human_color = "#7570B3"

In [ ]:
_scores = scores.pivot(columns="celltype_combination", index="cytokine", values="isin_mh_signature_frac")
_scores = _scores.loc[np.sum(_scores.fillna(0)>0.2, axis=1) > 0]
_scores = _scores.loc[:, np.sum(_scores.fillna(0)>0.2, axis=0) > 0]

mean_per_cytokine = _scores.mean(axis=1).sort_values(ascending=False)
order_y = mean_per_cytokine.index
order_x = _scores.mean(axis=0).sort_values(ascending=False).index

_scores = _scores.loc[order_y, order_x]

fig, ax = plt.subplots(1,4,figsize=(5.5, 3.5), sharey=False, width_ratios=(0.45, 0.45, 0.1, 0.1), gridspec_kw={})
ax_hm = ax[0]
ax_hm2 = ax[1]
ax_bar = ax[2]
ax_bar2 = ax[3]
# sns.heatmap(_scores, square=True, cbar=None, vmin=0, vmax=1, linewidth=0.5, linecolor="white", cmap="Oranges", ax=ax_hm, )
sns.heatmap(_scores, square=True, cbar=None, vmin=0, vmax=1, linewidth=0.5, linecolor="white", cmap="Greens", ax=ax_hm, )
ax_hm.set_facecolor("lightgray")
ax_hm.set_xlabel("")
ax_hm.set_ylabel("")
ax_hm.set_xticks(0.5+np.arange(_scores.shape[1]), [l.get_text().split("__")[1].replace("_", " ") for l in ax_hm.get_xticklabels()], fontsize=6, family="Arial")
ax_hm.set_yticks(0.5+np.arange(_scores.shape[0]), [replace_greek(l) for l in _scores.index], fontsize=6, family="Arial")
ax_hm.tick_params(length=0)
ax_hm.set_title("Overlap", fontsize=6, family="Arial")

_scores2 = scores.pivot(columns="celltype_combination", index="cytokine", values="pearson_signature_shared")
_scores2 = _scores2.loc[order_y, order_x]
sns.heatmap(_scores2, cmap="RdBu_r", ax=ax_hm2, cbar=False, square=True, vmin=-1, vmax=1, center=0, linewidth=0.5, linecolor="white")
ax_hm2.set_facecolor("lightgray")
ax_hm2.set_xlabel("")
ax_hm2.set_ylabel("")
ax_hm2.set_xticks(0.5+np.arange(_scores2.shape[1]), [l.get_text().split("__")[1].replace("_", " ") for l in ax_hm2.get_xticklabels()], fontsize=6, family="Arial")
# ax_hm2.set_yticks(0.5+np.arange(_scores2.shape[0]), [replace_greek(l) for l in _scores2.index], fontsize=6, family="Arial")
ax_hm2.set_yticks([])
ax_hm2.tick_params(length=0)
ax_hm2.set_title(r"$R$", fontsize=6, family="Arial")
box = ax_hm2.get_position()
box.x0 = box.x0 - 0.18
box.x1 = box.x1 - 0.18
ax_hm2.set_position(box)


box = ax_bar.get_position()
box.x0 = box.x0 - 0.28
box.x1 = box.x1 - 0.28
ax_bar.set_position(box)
sns.barplot(
    data=scores.loc[scores.cytokine.isin(order_y)],
    y="cytokine",
    x="num_signature_genes_m",
    ax=ax_bar,
    order=order_y,
    # color="gold",
    color = mouse_color,
    err_kws={'linewidth': 0.5},
)
sns.barplot(
    data=scores.loc[scores.cytokine.isin(order_y)],
    y="cytokine",
    x="num_signature_genes_m_homologs",
    ax=ax_bar,
    order=order_y,
    color="gray",
    err_kws={'linewidth': 0.5},
    alpha=0.6,
    errorbar=None,
)
ax_bar.set_xscale("log")
ax_bar.set_ylabel("")
ax_bar.set_xlabel("")
ax_bar.set_yticks([])
ax_bar.set_xticks([1, 10, 100, 1000], ["1", "10", "100", "1K"], fontsize=6, family="Arial", rotation=90)
ax_bar.set_xlim([0, 1100])
ax_bar.grid("y")
ax_bar.spines[['right', 'top']].set_visible(False)
ax_bar.set_xlabel("# DE genes\n(mouse)", fontsize=6, family="Arial")


box = ax_bar2.get_position()
box.x0 = box.x0 - 0.29
box.x1 = box.x1 - 0.29
ax_bar2.set_position(box)
sns.barplot(
    data=scores.loc[scores.cytokine.isin(order_y)],
    y="cytokine",
    x="num_signature_genes_h",
    ax=ax_bar2,
    order=order_y,
    color = human_color,
    err_kws={'linewidth': 0.5},
)
ax_bar2.set_xscale("log")
ax_bar2.set_ylabel("")
ax_bar2.set_xlabel("")
ax_bar2.set_yticks([])
ax_bar2.set_xticks([10, 100, 1000, 3000], ["10", "100", "1K", "3K"], fontsize=6, family="Arial", rotation=90)
# ax_bar2.set_xlim([0, 5300])
ax_bar2.grid("y")
ax_bar2.spines[['right', 'top']].set_visible(False)
ax_bar2.set_xlabel("# DE genes\n(human)", fontsize=6, family="Arial")

plt.savefig(os.path.join(FIGDIR, f"heatmap_dict_overlap_mh_homologs.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
norm = plt.Normalize(-1, 1)

sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
sm.set_array(np.array([]))

fig, ax = plt.subplots(1, 1, figsize=(0.7, 0.1))
cbar = plt.colorbar(sm, cax=ax, orientation="horizontal")

# cbar.set_label(r"$R$", rotation=0, family='Arial', fontsize=6)#labelpad=0, 
# cbar.ax.xaxis.set_ticks_position('top')
# cbar.ax.xaxis.set_label_position('top')
cbar.ax.tick_params(labelsize=6)
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)

for label in cbar.ax.get_yticklabels():
    label.set_fontfamily('Arial')

plt.tight_layout()
filename = f"colorbar_pearson_correlation.svg"
plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
norm = plt.Normalize(0, 1)

sm = plt.cm.ScalarMappable(cmap="Greens", norm=norm)
sm.set_array(np.array([]))

fig, ax = plt.subplots(1, 1, figsize=(0.7, 0.1))
cbar = plt.colorbar(sm, cax=ax, orientation="horizontal")

# cbar.set_label(r"$R$", rotation=0, family='Arial', fontsize=6)#labelpad=0, 
# cbar.ax.xaxis.set_ticks_position('top')
# cbar.ax.xaxis.set_label_position('top')
cbar.ax.tick_params(labelsize=6)
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)
cbar.ax.set_xticks([0, 0.5, 1], ["0%",  "50%", "100%"], family="Arial")

for label in cbar.ax.get_yticklabels():
    label.set_fontfamily('Arial')

plt.tight_layout()
filename = f"colorbar_overlap.svg"
plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(3.5, 3.5), sharey=False, width_ratios=(0.8, 0.2), gridspec_kw={})
ax_hm = ax[0]
_scores = scores.pivot(
    columns="celltype_combination",
    index="cytokine",
    values="pearson_signature_shared"
)
_scores = _scores.loc[order_y, order_x]
sns.heatmap(
    _scores,
    cmap="RdBu_r",
    ax=ax_hm,
    cbar=False,
    square=True,
    
)
ax_hm.set_facecolor("lightgray")
ax_hm.set_xlabel("")
ax_hm.set_ylabel("")
ax_hm.set_xticks(0.5+np.arange(_scores.shape[1]), [l.get_text().split("__")[1].replace("_", " ") for l in ax_hm.get_xticklabels()], fontsize=6, family="Arial")
ax_hm.set_yticks(0.5+np.arange(_scores.shape[0]), [replace_greek(l) for l in _scores.index], fontsize=6, family="Arial")
ax_hm.tick_params(length=0)
plt.show()

In [ ]:
_scores = scores.pivot(
    columns="celltype_combination",
    index="cytokine",
    values="isin_hm_signature_frac"
)
_scores = _scores.loc[np.sum(_scores.fillna(0)>0.05, axis=1) > 0]
_scores = _scores.loc[:, np.sum(_scores.fillna(0)>0.05, axis=0) > 1]
print(_scores.max())
fig, ax = plt.subplots(1,1,figsize=(1.8, 3.5))
sns.heatmap(_scores, square=True, cbar=None, cmap="Reds", vmin=0, vmax=1, linewidth=0.5, linecolor="white")
ax.set_facecolor("lightgray")
ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xticks(0.5+np.arange(_scores.shape[1]), [l.get_text().split("__")[1].replace("_", " ") for l in ax.get_xticklabels()], fontsize=6, family="Arial")
ax.set_yticks(0.5+np.arange(_scores.shape[0]), [replace_greek(l) for l in _scores.index], fontsize=6, family="Arial")
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="isin_mh_signature_frac",
    y="isin_hm_signature_frac",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="num_signature_genes_h",
    y="isin_hm_signature_frac",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="num_signature_genes_h",
    y="isin_hm_signature",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="num_signature_genes_m",
    y="isin_mh_signature",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="num_signature_genes_m_homologs",
    y="isin_mh_signature",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="num_signature_genes_m",
    y="isin_mh_signature",
)
sns.scatterplot(
    data=scores,
    x="num_signature_genes_m_homologs",
    y="isin_mh_signature",
    alpha=0.3,
)
plt.show()

In [ ]:
scores.groupby("celltype_m").isin_mh_signature_frac.mean().sort_values(ascending=False)

In [ ]:
scored_mean_per_cytokine = scores.groupby("cytokine").isin_mh_signature_frac.mean().sort_values(ascending=False)
scored_mean_per_cytokine = scored_mean_per_cytokine.loc[scored_mean_per_cytokine>0.1]
order = scored_mean_per_cytokine.index

import matplotlib.cm as cm
mean_num_genes_per_cytokine = dict(scores.groupby("cytokine").num_signature_genes_m_homologs.mean().sort_values())
values = list(mean_num_genes_per_cytokine.values())
norm = plt.Normalize(min(values), max(values))
colors = [cm.Oranges(norm(v)) for v in values]
cmap = dict(zip(mean_num_genes_per_cytokine.keys(), colors))

fig, ax = plt.subplots(1,1,figsize=(2.5, 1.3))
sns.barplot(
    data=scores,
    x="cytokine",
    y="isin_mh_signature_frac",
    hue="cytokine",
    order=order,
    legend=False,
    palette=cmap,
    err_kws={'linewidth': 0.5},
    edgecolor='black',
    linewidth=0.5,
)
yticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(l) for l in yticks], fontsize=6, family="Arial", rotation=0)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), fontsize=6, family="Arial", rotation=90)
ax.spines[['right', 'top']].set_visible(False)
y_margin=0.05
ax.margins(y=y_margin)
plt.xlabel("")
plt.ylabel("")
plt.grid(axis="y")
plt.savefig(os.path.join(FIGDIR, f"heatmap_dict_overlap_mh_per_celltype.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores,
    x="num_signature_genes_m",
    y="num_signature_genes_m_homologs",
    ax=ax,
)
xlims = plt.gca().get_xlim()
ylims = plt.gca().get_ylim()
plt.plot(np.linspace(xlims[0], xlims[1]), np.linspace(ylims[0], ylims[1]), "--k")
plt.show()

In [ ]:
cut = np.inf
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores.loc[scores.num_signature_genes_m < cut],
    x="num_signature_genes_m",
    y="num_signature_genes_h",
    ax=ax,
)
xlims = plt.gca().get_xlim()
ylims = plt.gca().get_ylim()
plt.xscale("log")
plt.yscale("log")
plt.plot(np.linspace(xlims[0], xlims[1]), np.linspace(ylims[0], ylims[1]), "--k")
plt.xlabel("# signature genes in mouse dict")
plt.ylabel("# signature genes in human dict")
plt.show()
print(f"pearson: {pearsonr(scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_m, scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_h)[0]}")
print(f"spearman: {spearmanr(scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_m, scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_h)[0]}")

cut = 100
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores.loc[scores.num_signature_genes_m < cut],
    x="num_signature_genes_m",
    y="num_signature_genes_h",
    ax=ax,
)
xlims = plt.gca().get_xlim()
ylims = plt.gca().get_ylim()
plt.plot(np.linspace(xlims[0], xlims[1]), np.linspace(ylims[0], ylims[1]), "--k")
plt.xlabel("# signature genes in mouse dict")
plt.ylabel("# signature genes in human dict")
plt.show()
print(f"pearson: {pearsonr(scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_m, scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_h)[0]}")
print(f"spearman: {spearmanr(scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_m, scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_h)[0]}")


cut = 25
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores.loc[scores.num_signature_genes_m < cut],
    x="num_signature_genes_m",
    y="num_signature_genes_h",
    ax=ax,
)
xlims = plt.gca().get_xlim()
ylims = plt.gca().get_ylim()
plt.plot(np.linspace(xlims[0], xlims[1]), np.linspace(ylims[0], ylims[1]), "--k")
plt.xlabel("# signature genes in mouse dict")
plt.ylabel("# signature genes in human dict")
plt.show()
print(f"pearson: {pearsonr(scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_m, scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_h)[0]}")
print(f"spearman: {spearmanr(scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_m, scores.loc[scores.num_signature_genes_m < cut].num_signature_genes_h)[0]}")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores, 
    x="num_signature_genes_shared",
    y="pearson_signature_shared",
    ax=ax,
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,3))
sns.scatterplot(
    data=scores, 
    x="num_all_genes_shared",
    y="pearson_all_shared",
    ax=ax,
)
plt.show()

In [ ]:
mcd